# 1. Importing data
## 1.1. Libraries

In [1]:
import json, os
import transport_engine as tm
import pandas as pd

## 1.2. Instances

*thesis_instances.json* file contains the information required to extract **100** instances. This information helps to differentiate between instances. Below, instances' infomation is saved to a list called *instances*. 

In [2]:
file_path = "thesis_instances.json"
if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        instances = json.load(f)

Next step is to read instances. Below, you can find how simple you can read a instance with *transport_engine* functions. *load_instance* is the function to import an instance that takes a single argument of a dictionary with required instance information. You can run *?tm.load_instance* to see more details.


In [3]:
?tm.load_instance

Signature: tm.load_instance(instance)
Docstring:
This function helps to laod instances.
This requires a single argument which is a dictionary with keys:
* map : folder name that contains cities, airway, highway, vehicle data; and fleet and load scenarios folders
* fleet : fleet file (must exist in fleet_scenarios file)
* loads : loads file (must exist in loads_scenarios file)
* broken_arcs : list of broken highway arcs
File:      ~/Desktop/Researches/Multimodular_transportation/Code/transport_engine.py
Type:      function

Below, first instance in *instances* list is imported. It returns **6** dataframes.
* cities
* highway
* airway
* vehicle
* fleet
* loads

In [4]:
cities, highway, airway, vehicle, fleet, loads = tm.load_instance(instances[0])

Of course, this dataframes are only for the first instance, you can utilize this line of code in an iterative manner.

You can check each dataframe, e.g., by looking into the head rows and columns names. This helps to get familiar with data. Below, you are seeing first 5 rows of **cities** dataframe.

In [5]:
cities.head(5)

,name,coordinate_x,coordinate_y,airport
id,,,,
1,sah,7.06,5.96,False
2,kmy,1.31,4.62,False
3,rnj,9.69,2.26,False
4,aye,11.85,2.90,True
5,iig,18.22,3.87,False


# 3. Solution Format and Deliverables

## 3.1. Solution Files

I request you to submit your solutions as three dataframes. These are:

* **routes**: This dataframe contains generic information regarding to each route you have created. It must consist of minimum of four rows -you are free to have additional columns if they help you in developing your algorithm.
    * **route_id**: Unique route id.
    * **type**: *'highway'* or *'airway'*. Note that any value rather than these two will arise error.
    * **vehicle_id**: Id of the vehicle that execute this route. This corresponds to *id* of *fleet* dataframe.
    * **loads**: Ids of loads that are carried over with this route. This corresponds to *id* of *loads* dataframe.

* **route_arcs**: This dataframe contains movements of routes. This dataframe is directly related to *routes* dataframe.
    * **route_id**: Id of related route. This corresponds to *id* of *route* dataframe from your solution, so they must be consistent.
    * **order**: Execution order of the movement in the route.
    * **arc_id**: Id of the arcs traveled. This corresponds to *id* of either *highway* or *airway* dataframe.
    * **start_time**: The time that the vehicle starts traveling the corresponding arcs.
    * **end_time**: The time that the vehicle finishes traveling the corresponding arcs. Note that, you must take vehicle's speed into consideration to calculate the travel time.
    * **loads_on**: A list that contains the loads that are on vehicle while traveling the corresponding arc. If the vehicle is empty, then place an empty list to this column instead of leaving it empty.

* **vehicle_schedule**: This dataframe contains schedule of vehicles. Note that, this information can directly be extracted from previous two tables *routes* and *route_arcs*. Therefore, consistency is the key while building this dataframe. This also helps you to understand relational databases, which is widely utilized especially in big companies. This also helpful if you need to schedule a vehicle more than once even though it is unlikely in the current setting (why?).
    * **vehicle_id**: Id of the vehicle that execute this route. This corresponds to *id* of *fleet* dataframe.
    * **route_id**: Id of related route. This corresponds to *id* of *route* dataframe from your solution.
    * **start_time**: Start time of the given route.
    * **end_time**: End time of the given route.

You may start with below code to initialize your solution dataframes. 

In [6]:
routes = {'route_id' : [],
          'type' : [],
          'vehicle_id' : [],
          'loads' : []}

route_arcs = {'route_id' : [],
              'order' : [],
              'arc_id' : [],
              'start_time' : [],
              'end_time' : [],
              'loads_on' : []}

vehicle_schedule = {"vehicle_id": [],
                    "route_id": [],
                    "start_time" : [],
                    "end_time" : []}

routes_df = pd.DataFrame(routes).set_index('route_id')
moves_df = pd.DataFrame(route_arcs)
vehicle_schedule_df = pd.DataFrame(vehicle_schedule)

## 3.2. Example Solution and Validator

Below, an example solution is provided. Note that, it is not a complete solution, it only creates a route for the first load.

In [7]:
#AN EXAMPLE OF ROUTE
route_id = 1
routes['route_id'].append(route_id)
routes['type'].append('highway')
routes['vehicle_id'].append(7)
routes['loads'].append([1])

# ARCS OF ROUTE
route_arcs['route_id'].append(route_id)
route_arcs['order'].append(1)
route_arcs['arc_id'].append(26)
route_arcs['start_time'].append(0) #15.27
route_arcs['end_time'].append(15.27)
route_arcs['loads_on'].append([1])

route_arcs['route_id'].append(route_id)
route_arcs['order'].append(2)
route_arcs['arc_id'].append(27)
route_arcs['start_time'].append(16)
route_arcs['end_time'].append(31.27)
route_arcs['loads_on'].append([])

#SAME RECORD IN VEHICLE SCHEDULE
vehicle_schedule['vehicle_id'].append(7)
vehicle_schedule['route_id'].append(1)
vehicle_schedule['start_time'].append(0)
vehicle_schedule['end_time'].append(31.27)

In our problem, one of the main challenges is not to carry loads directly, but in some cases it might be beneficial (or sometimes necessary) to use multiple transportation types. In this case, you must consider transfering conditions. Below, I provide another example in which a load is carried with routes of different types, e.g., highway and airway. 

Below, load 4 is carried with two operations. First, it is carried from location 2 to 6 in highway (route 2), and then its logictics continue in airway, and it is taken location 4,its final destination (route 3).

In [8]:
# MULTI MODAL EXAMPLE
route_id = 2
routes['route_id'].append(route_id)
routes['type'].append('highway')
routes['vehicle_id'].append(5)
routes['loads'].append([4])

# ARCS OF ROUTE
route_arcs['route_id'].append(route_id)
route_arcs['order'].append(1)
route_arcs['arc_id'].append(16)
route_arcs['start_time'].append(0)
route_arcs['end_time'].append(3.55)
route_arcs['loads_on'].append([4])

route_arcs['route_id'].append(route_id)
route_arcs['order'].append(2)
route_arcs['arc_id'].append(17)
route_arcs['start_time'].append(3.55)
route_arcs['end_time'].append(7.1)
route_arcs['loads_on'].append([])

#SAME RECORD IN VEHICLE SCHEDULE
vehicle_schedule['vehicle_id'].append(5)
vehicle_schedule['route_id'].append(route_id)
vehicle_schedule['start_time'].append(0)
vehicle_schedule['end_time'].append(7.1)


route_id = 3
routes['route_id'].append(route_id)
routes['type'].append('airway')
routes['vehicle_id'].append(1)
routes['loads'].append([4])

# ARCS OF ROUTE
route_arcs['route_id'].append(route_id)
route_arcs['order'].append(1)
route_arcs['arc_id'].append(1)
route_arcs['start_time'].append(3.55)
route_arcs['end_time'].append(4.91)
route_arcs['loads_on'].append([4])

route_arcs['route_id'].append(route_id)
route_arcs['order'].append(2)
route_arcs['arc_id'].append(0)
route_arcs['start_time'].append(4.91)
route_arcs['end_time'].append(6.27)
route_arcs['loads_on'].append([])

#SAME RECORD IN VEHICLE SCHEDULE
vehicle_schedule['vehicle_id'].append(1)
vehicle_schedule['route_id'].append(route_id)
vehicle_schedule['start_time'].append(3.55)
vehicle_schedule['end_time'].append(6.27)

After converting your routes into dataframes, I also provide you *validator* function of transport_engine module to check feasibility of your output. It returns three objects:
* **messages**: This is a list of warnings, errors, and feasibility checks. An correct output must have an empty list, so you should take care of every messages.
* **route_costs**: A dictionary with keys being route id and values being calculated cost of it
* **penalties**: A dictionary with keys being load id and values being calculated penalty for it

In [9]:
vehicle_schedule_df

,vehicle_id,route_id,start_time,end_time


In [10]:
routes_df = pd.DataFrame(routes).set_index('route_id')
moves_df = pd.DataFrame(route_arcs)
vehicle_schedule_df = pd.DataFrame(vehicle_schedule)


messages, route_costs, penalties = tm.validator(instances[0], routes_df, moves_df, vehicle_schedule_df)

for message in messages:
    print(message)
print(route_costs)
print(penalties)

Load 2 is not carried over with any of the routes
Load 3 is not carried over with any of the routes
Load 5 is not carried over with any of the routes
Load 6 is not carried over with any of the routes
Load 7 is not carried over with any of the routes
Load 8 is not carried over with any of the routes
Load 9 is not carried over with any of the routes
Load 10 is not carried over with any of the routes
Load 11 is not carried over with any of the routes
Load 12 is not carried over with any of the routes
Load 13 is not carried over with any of the routes
Load 14 is not carried over with any of the routes
Load 15 is not carried over with any of the routes
Load 16 is not carried over with any of the routes
Load 17 is not carried over with any of the routes
Load 18 is not carried over with any of the routes
Load 19 is not carried over with any of the routes
Load 20 is not carried over with any of the routes
Load 21 is not carried over with any of the routes
Load 22 is not carried over with any o

Ensure that your solutions do not have any error message. Note that, all messages are **NOT NECESSARILY** valid. Therefore, you should always double check your solution with the messages. **If you disagree with any of the messages, please let me know soon, so we can check there is really an error.**

## 3.3. Deliverables

You should create a solution folder to deliver your solutions -you can name this main folder freely. This folder should have other folders with the name of *instance_id*, e.g., **inst_001**. Under this folder, you should write *routes*, *route_arcs*, and *vehicle_schedule* dataframes as csv files such as **routes.csv**. You can utilize below a piece of code to write your solutions for each instance. You can also iterate over instances.

In [11]:
# solution_folder = 'SOLUTIONS' # you may have a different name, e.g. with your last name.
# if solution_folder not in os.listdir():
#     os.mkdir(solution_folder)

# #for instances[0]
# solution_route = f"{solution_folder}/{instances[0]['instance_id']}" 
# if f"{instances[0]['instance_id']}" not in os.listdir(solution_folder):
#     os.mkdir(solution_route)
# routes_df.to_csv(f"{solution_route}/routes.csv")
# moves_df.to_csv(f"{solution_route}/moves.csv")
# vehicle_schedule_df.to_csv(f"{solution_route}/vehicle_schedule.csv")